In [2]:
import ROOT as R
import pandas as pd

data_files = "~/hps/track_cluster_matching/data/*.root"
mc_files = "~/hps/ecal_calibration/data/fee_mc/*.root"

df_data = R.RDataFrame("MiniDST", data_files)
df_mc = R.RDataFrame("MiniDST", mc_files)

print("Events in data:", df_data.Count().GetValue())
print("Events in MC:", df_mc.Count().GetValue())

Events in data: 1918784
Events in MC: 617450


In [ ]:
# Plot mean or sum energy maps in x-y bins, for data and MC, side by side.

def mean_energy_map(df, name, title,
                    nx=120, xmin=-300, xmax=300,
                    ny=60,  ymin=-150, ymax=150):
    """
    Returns (h_mean, h_sumE, h_count) where:
      h_sumE(x,y)  = sum of cluster energies in bin
      h_count(x,y) = number of clusters in bin
      h_mean(x,y)  = h_sumE / h_count
    Works with vector branches: ecal_cluster_x/y/energy (RVec).
    """

    # weight=1 vector with same length as clusters, so Histo2D can count entries
    df2 = df.Define("clus_one", "ROOT::VecOps::RVec<float>(ecal_cluster_energy.size(), 1.0f)")

    h_sumE = df2.Histo2D(
        (f"h_sumE_{name}", f"{title}; ECAL x [mm]; ECAL y [mm]", nx, xmin, xmax, ny, ymin, ymax),
        "ecal_cluster_x", "ecal_cluster_y", "ecal_cluster_energy"
    )

    h_cnt = df2.Histo2D(
        (f"h_cnt_{name}", f"{title} (counts); ECAL x [mm]; ECAL y [mm]", nx, xmin, xmax, ny, ymin, ymax),
        "ecal_cluster_x", "ecal_cluster_y", "clus_one"
    )

    # Trigger event loop once
    hs = h_sumE.GetPtr()
    hc = h_cnt.GetPtr()

    h_mean = hs.Clone(f"h_meanE_{name}")
    h_mean.SetTitle(f"{title} (mean cluster energy); ECAL x [mm]; ECAL y [mm]")
    h_mean.Divide(hc)  # mean = sum / count
    return h_mean, hs, hc


# --- build maps ---
h_mean_data, h_sum_data, h_cnt_data = mean_energy_map(df_data, "data", "DATA")
h_mean_mc,   h_sum_mc,   h_cnt_mc   = mean_energy_map(df_mc,   "mc",   "FEE MC")

# --- draw side-by-side ---
c = R.TCanvas("c_ecal_maps", "ECAL mean energy maps", 1400, 650)
c.Divide(2,1)

c.cd(1); R.gPad.SetGrid()
h_mean_data.Draw("COLZ")
# h_sum_data.Draw("COLZ")

c.cd(2); R.gPad.SetGrid()
h_mean_mc.Draw("COLZ")
# h_sum_mc.Draw("COLZ")
c.Draw()
# c.SaveAs("plots/ecal_mean_energy_maps_data_vs_mc.png")

Warning in <TCanvas::Constructor>: Deleting canvas with same name: c_ecal_maps


In [ ]:
# Pick ONE crystal to start (you can change these)
SEED_IX = -5
SEED_IY = -3

# Cluster selection from slides (Etot>2, Eseed/Etot>0.6)
df_mc_sel = (
    df_mc
    .Filter("ecal_cluster_energy.size() > 0")
    .Define("seed_over_etot", "ecal_cluster_seed_energy / (ecal_cluster_energy + 1e-9f)")
    .Define("mask",
            f"(ecal_cluster_seed_ix == {SEED_IX})"
            f" && (ecal_cluster_seed_iy == {SEED_IY})"
            f" && (ecal_cluster_energy > 2.0f)"
            f" && (seed_over_etot > 0.6f)")
    .Filter("Any(mask)")
    .Define("Esel", "ecal_cluster_energy[mask]")
)

hE = df_mc_sel.Histo1D(
    ("hE", f"FEE MC cluster energy (seed ix,iy=({SEED_IX},{SEED_IY})); E [GeV]; Counts",
     300, 2.0, 4.5),
    "Esel"
)
h = hE.GetPtr()

c = R.TCanvas("c","FEE MC peak",800,600)
c.SetGrid()
h.SetLineWidth(2)
h.Draw("HIST")

# Fit the peak (adjust window after first look)
peak_bin = h.GetMaximumBin()
peak_x = h.GetXaxis().GetBinCenter(peak_bin)
fit_lo, fit_hi = peak_x - 0.2, peak_x + 0.2

f = R.TF1("f","gaus", fit_lo, fit_hi)
h.Fit(f, "RQ")

mu_mc = f.GetParameter(1)
sig_mc = abs(f.GetParameter(2))
print(f"EFEE_MC seed({SEED_IX},{SEED_IY}) = {mu_mc:.4f} GeV  (sigma={sig_mc:.4f} GeV)")

f.SetLineColor(R.kRed); f.SetLineWidth(2); f.Draw("SAME")
c.Draw()

EFEE_MC seed(-23,-5) = 2.0500 GeV  (sigma=0.0509 GeV)


In [15]:
SEED_IX = -5
SEED_IY = -3
PSUM_MIN = 0.3

# --- MC selection (unchanged) ---
df_mc_sel = (
    df_mc
    .Filter("ecal_cluster_energy.size() > 0")
    .Define("seed_over_etot", "ecal_cluster_seed_energy / (ecal_cluster_energy + 1e-9f)")
    .Define("mask",
            f"(ecal_cluster_seed_ix == {SEED_IX})"
            f" && (ecal_cluster_seed_iy == {SEED_IY})"
            f" && (ecal_cluster_energy > 2.0f)"
            f" && (seed_over_etot > 0.6f)")
    .Filter("Any(mask)")
    .Define("Esel", "ecal_cluster_energy[mask]")
)

# --- Data selection: FEE electrons (pdg==11, psum > PSUM_MIN) matched to this seed ---
df_data_sel = (
    df_data
    .Filter("ecal_cluster_energy.size() > 0 && part_pdg.size() > 0")
    .Define("seed_over_etot", "ecal_cluster_seed_energy / (ecal_cluster_energy + 1e-9f)")
    .Define("clus_mask",
            f"(ecal_cluster_seed_ix == {SEED_IX})"
            f" && (ecal_cluster_seed_iy == {SEED_IY})"
            f" && (ecal_cluster_energy > 2.0f)"
            f" && (seed_over_etot > 0.6f)")
    .Define("Esel", f"""
        ROOT::VecOps::RVec<float> out;
        for (size_t i = 0; i < part_pdg.size(); ++i) {{
            if (part_pdg[i] != 11) continue;
            int cl = part_ecal_cluster[i];
            if (cl < 0 || cl >= (int)ecal_cluster_energy.size()) continue;
            if (!clus_mask[cl]) continue;
            float px = part_px[i], py = part_py[i], pz = part_pz[i];
            float psum = std::sqrt(px*px + py*py + pz*pz);
            if (psum < {PSUM_MIN}f) continue;
            out.push_back(ecal_cluster_energy[cl]);
        }}
        return out;
    """)
    .Filter("Esel.size() > 0")
)

# --- Histograms ---
hMC = df_mc_sel.Histo1D(
    ("hMC", f"FEE cluster energy seed({SEED_IX},{SEED_IY}); E [GeV]; Counts", 300, 2.0, 4.5),
    "Esel"
).GetPtr()

hData = df_data_sel.Histo1D(
    ("hData", f"FEE cluster energy seed({SEED_IX},{SEED_IY}); E [GeV]; Counts", 300, 2.0, 4.5),
    "Esel"
).GetPtr()

# --- Normalize data to MC peak height for visual comparison ---
if hData.GetMaximum() > 0:
    hData.Scale(hMC.GetMaximum() / hData.GetMaximum())

# --- Fit both ---
def fit_gaus(h, color):
    peak_x = h.GetXaxis().GetBinCenter(h.GetMaximumBin())
    f = R.TF1(f"f_{h.GetName()}", "gaus", peak_x - 0.2, peak_x + 0.2)
    h.Fit(f, "RQ0")
    f.SetLineColor(color)
    f.SetLineWidth(2)
    return f, f.GetParameter(1), abs(f.GetParameter(2))

fMC,   mu_mc,   sig_mc   = fit_gaus(hMC,   R.kRed)
fData, mu_data, sig_data = fit_gaus(hData, R.kBlue)

print(f"MC   seed({SEED_IX},{SEED_IY}): mu={mu_mc:.4f} GeV  sigma={sig_mc:.4f} GeV")
print(f"Data seed({SEED_IX},{SEED_IY}): mu={mu_data:.4f} GeV  sigma={sig_data:.4f} GeV")

# --- Plot ---
c = R.TCanvas("c", "FEE peak MC vs Data", 800, 600)
c.SetGrid()

hMC.SetLineColor(R.kBlack); hMC.SetLineWidth(2)
hData.SetLineColor(R.kBlue); hData.SetLineWidth(2)

hMC.Draw("HIST")
hData.Draw("HIST SAME")
fMC.Draw("SAME")
fData.Draw("SAME")

leg = R.TLegend(0.6, 0.7, 0.88, 0.88)
leg.AddEntry(hMC,   f"MC (mu={mu_mc:.3f})",     "l")
leg.AddEntry(hData, f"Data (mu={mu_data:.3f})",  "l")
leg.Draw()

c.Draw()

MC   seed(-5,-3): mu=3.7336 GeV  sigma=0.0715 GeV
Data seed(-5,-3): mu=3.2233 GeV  sigma=0.2912 GeV


In [19]:
# Get unique (ix,iy) pairs from MC by taking a small sample of clusters
# (Increase Range if it misses crystals)
df_pairs = (
    df_mc
    .Filter("ecal_cluster_seed_ix.size() > 0")
    .Range(200)  # speed-up; increase later
    .Define("ix", "ecal_cluster_seed_ix")
    .Define("iy", "ecal_cluster_seed_iy")
)

# Snapshot ix/iy vectors, then build unique pairs in Python
tmp = df_pairs.AsNumpy(["ix","iy"])
# Look at first event
print("Event 0 ix:", list(tmp["ix"][1]))
print("Event 0 iy:", list(tmp["iy"][1]))
pairs = set()
for vix, viy in zip(tmp["ix"], tmp["iy"]):
    for a,b in zip(vix, viy):
        pairs.add((int(a), int(b)))

pairs = sorted(pairs)
print("Found seed crystals:", len(pairs))
print("First 10:", pairs)

Event 0 ix: [19]
Event 0 iy: [1]
Found seed crystals: 157
First 10: [(-23, -4), (-23, -1), (-23, 1), (-22, 1), (-22, 3), (-21, -2), (-21, 1), (-21, 2), (-21, 4), (-20, 2), (-19, -3), (-19, -2), (-19, -1), (-19, 2), (-18, -2), (-18, 4), (-17, -2), (-17, 1), (-16, -3), (-16, -2), (-16, 1), (-16, 3), (-16, 4), (-16, 5), (-15, -4), (-15, -2), (-15, -1), (-15, 1), (-14, -3), (-14, -1), (-14, 3), (-14, 4), (-13, -5), (-13, -4), (-13, -2), (-13, -1), (-12, 4), (-12, 5), (-11, -3), (-11, -2), (-11, 1), (-11, 3), (-10, -3), (-10, 3), (-9, -4), (-9, -2), (-9, 3), (-8, -2), (-8, 3), (-7, -5), (-7, -3), (-7, 5), (-6, -4), (-6, -3), (-6, -2), (-6, 3), (-6, 5), (-5, -4), (-5, 2), (-4, -3), (-4, 2), (-4, 3), (-4, 5), (-3, 2), (-3, 3), (-3, 4), (-2, -4), (-2, -3), (-2, 2), (-1, -3), (-1, -1), (-1, 3), (1, 1), (1, 2), (1, 3), (1, 4), (2, -3), (2, 3), (3, -4), (3, -3), (3, -2), (3, -1), (3, 1), (3, 3), (3, 4), (4, -4), (4, -3), (4, -2), (4, 1), (4, 3), (4, 4), (5, -3), (5, -2), (5, -1), (6, -3), (6, -2)

In [5]:
def fit_fee_peak_for_seed(df, seed_ix, seed_iy, tag,
                          Emin=2.0, seedFracMin=0.6,
                          nbins=250, xlo=2.5, xhi=4.5,
                          fit_halfwidth=0.2, min_entries=200):

    d = (df
        .Filter("ecal_cluster_energy.size() > 0")
        .Define("seed_over_etot", "ecal_cluster_seed_energy / (ecal_cluster_energy + 1e-9f)")
        .Define("mask",
                f"(ecal_cluster_seed_ix == {seed_ix})"
                f" && (ecal_cluster_seed_iy == {seed_iy})"
                f" && (ecal_cluster_energy > {Emin}f)"
                f" && (seed_over_etot > {seedFracMin}f)")
        .Filter("Any(mask)")
        .Define("Esel", "ecal_cluster_energy[mask]")
    )

    hR = d.Histo1D((f"hE_{tag}_{seed_ix}_{seed_iy}",
                    f"Eclus seed({seed_ix},{seed_iy}) [{tag}]; E [GeV]; Counts",
                    nbins, xlo, xhi),
                   "Esel")
    h = hR.GetPtr()

    if h.GetEntries() < min_entries:
        return None  # too few stats

    peak_bin = h.GetMaximumBin()
    peak_x = h.GetXaxis().GetBinCenter(peak_bin)

    f = R.TF1(f"f_{tag}_{seed_ix}_{seed_iy}", "gaus",
              peak_x - fit_halfwidth, peak_x + fit_halfwidth)

    h.Fit(f, "RQ0")  # quiet, range, don't draw

    mu = f.GetParameter(1)
    sig = abs(f.GetParameter(2))
    return float(mu), float(sig), int(h.GetEntries())

In [13]:
def fit_fee_peak_for_seed(df, seed_ix, seed_iy, tag,
                          # cluster-quality (from slides)
                          Emin=2.0, seedFracMin=0.6,
                          # data FEE-like track selection (only if apply_fee_track_sel=True)
                          apply_fee_track_sel=False,
                          pmin=3.3, eop_min=0.8, eop_max=1.2,
                          # histogram/fit controls
                          nbins=250, xlo=2.5, xhi=4.5,
                          fit_halfwidth=0.2, min_entries=200):

    # --- shared: cluster-quality mask on *all clusters* in event
    base = (df
        .Filter("ecal_cluster_energy.size() > 0")
        .Define("seed_over_etot", "ecal_cluster_seed_energy / (ecal_cluster_energy + 1e-9f)")
        .Define("clus_mask",
                f"(ecal_cluster_seed_ix == {seed_ix})"
                f" && (ecal_cluster_seed_iy == {seed_iy})"
                f" && (ecal_cluster_energy > {Emin}f)"
                f" && (seed_over_etot > {seedFracMin}f)")
    )

    if not apply_fee_track_sel:
        # MC FEE case: just histogram qualifying clusters in that seed
        dsel = (base
            .Filter("Any(clus_mask)")
            .Define("Esel", "ecal_cluster_energy[clus_mask]")
        )
    else:
        # DATA case: require FEE-like electron particle that matches THIS seed crystal
        # We build a particle-level mask that ties the particle's matched cluster to the seed (ix,iy)
        dsel = (base
            .Filter("part_pdg.size()>0")
            .Define("fee_part_mask", f"""
                ROOT::VecOps::RVec<char> m(part_pdg.size(), 0);
                for (size_t i=0; i<part_pdg.size() ; ++i){{
                    if (part_pdg[i] != 11) continue;              // FEE = electron only
                    int tr = part_track[i];
                    int cl = part_ecal_cluster[i];
                    if (tr < 0 || cl < 0) continue;
                    if (tr >= (int)track_px.size() || cl >= (int)ecal_cluster_energy.size()) continue;

                    // require the matched cluster is THIS seed crystal and passes cluster-quality mask
                    if (!(ecal_cluster_seed_ix[cl] == {seed_ix} && ecal_cluster_seed_iy[cl] == {seed_iy})) continue;
                    if (!clus_mask[cl]) continue;

                    // beam-like momentum + E/p window
                    float px = track_px[tr], py = track_py[tr], pz = track_pz[tr];
                    float mom = std::sqrt(px*px + py*py + pz*pz);
                    if (mom < {pmin}f) continue;

                    float E = ecal_cluster_energy[cl];
                    float eop = E / (mom + 1e-9f);
                    if (eop < {eop_min}f || eop > {eop_max}f) continue;

                    m[i] = 1;
                }}
                return m;
            """)
            .Filter("Any(fee_part_mask)")
            # Now collect E of the matched cluster for those passing electrons
            .Define("Esel", f"""
                ROOT::VecOps::RVec<float> out;
                for (size_t i=0; i<fee_part_mask.size(); ++i){{
                    if (!fee_part_mask[i]) continue;
                    int cl = part_ecal_cluster[i];
                    if (cl < 0 || cl >= (int)ecal_cluster_energy.size()) continue;
                    out.push_back(ecal_cluster_energy[cl]);
                }}
                return out;
            """)
        )

    # histogram + fit
    hR = dsel.Histo1D((f"hE_{tag}_{seed_ix}_{seed_iy}",
                       f"Eclus seed({seed_ix},{seed_iy}) [{tag}]; E [GeV]; Counts",
                       nbins, xlo, xhi),
                      "Esel")
    h = hR.GetPtr()

    if h.GetEntries() < min_entries:
        return None

    peak_bin = h.GetMaximumBin()
    peak_x = h.GetXaxis().GetBinCenter(peak_bin)
    f = R.TF1(f"f_{tag}_{seed_ix}_{seed_iy}", "gaus", peak_x - fit_halfwidth, peak_x + fit_halfwidth)
    h.Fit(f, "RQ0")

    mu = f.GetParameter(1)
    sig = abs(f.GetParameter(2))
    return float(mu), float(sig), int(h.GetEntries())

In [20]:
pairs = [(-23, -4), (-23, -1), (-23, 1), (-22, 1), (-22, 3), (-21, -2), (-21, 1), (-21, 2), (-21, 4), (-20, 2), (-19, -3), (-19, -2), (-19, -1), (-19, 2), (-18, -2), (-18, 4), (-17, -2), (-17, 1), (-16, -3), (-16, -2), (-16, 1), (-16, 3), (-16, 4), (-16, 5), (-15, -4), (-15, -2), (-15, -1), (-15, 1), (-14, -3), (-14, -1), (-14, 3), (-14, 4), (-13, -5), (-13, -4), (-13, -2), (-13, -1), (-12, 4), (-12, 5), (-11, -3), (-11, -2), (-11, 1), (-11, 3), (-10, -3), (-10, 3), (-9, -4), (-9, -2), (-9, 3), (-8, -2), (-8, 3), (-7, -5), (-7, -3), (-7, 5), (-6, -4), (-6, -3), (-6, -2), (-6, 3), (-6, 5), (-5, -4), (-5, 2), (-4, -3), (-4, 2), (-4, 3), (-4, 5), (-3, 2), (-3, 3), (-3, 4), (-2, -4), (-2, -3), (-2, 2), (-1, -3), (-1, -1), (-1, 3), (1, 1), (1, 2), (1, 3), (1, 4), (2, -3), (2, 3), (3, -4), (3, -3), (3, -2), (3, -1), (3, 1), (3, 3), (3, 4), (4, -4), (4, -3), (4, -2), (4, 1), (4, 3), (4, 4), (5, -3), (5, -2), (5, -1), (6, -3), (6, -2), (6, 1), (6, 3), (7, -2), (7, -1), (7, 1), (7, 4), (8, -3), (8, -2), (8, -1), (8, 3), (8, 4), (8, 5), (9, -1), (9, 1), (10, -4), (10, -1), (10, 1), (10, 2), (10, 5), (11, -4), (11, 3), (12, -4), (12, 2), (12, 3), (12, 4), (13, -5), (13, -1), (13, 1), (13, 3), (14, -2), (14, 1), (15, -3), (15, -2), (15, 1), (15, 2), (15, 3), (16, -5), (16, -1), (16, 2), (17, -2), (17, 1), (17, 2), (18, -5), (18, -4), (18, 1), (18, 2), (19, -4), (19, 1), (19, 2), (19, 5), (20, -3), (20, 1), (21, -5), (21, -3), (21, 1), (21, 2), (21, 3), (21, 4), (22, -2), (22, 5), (23, -5)]

results = []

# Example: do first 200 crystals to start (so you can sanity check fast)
for (ix, iy) in pairs[:20]:
    r_mc = fit_fee_peak_for_seed(df_mc, ix, iy, "MC")
    r_da = fit_fee_peak_for_seed(df_data, ix, iy, "DATA", apply_fee_track_sel=True)

    if (r_mc is None) or (r_da is None):
        continue

    mu_mc, sig_mc, n_mc = r_mc
    mu_da, sig_da, n_da = r_da

    C = mu_mc / mu_da if mu_da != 0 else None
    results.append((ix, iy, mu_mc, sig_mc, n_mc, mu_da, sig_da, n_da, C))

print("Crystals with fits:", len(results))
print("First 5 rows:")
for row in results:
    print(row)

Crystals with fits: 0
First 5 rows:
